# Gold - fact_equipment_telemetry_hourly

This notebook loads `amperedrive_giga.gold.fact_equipment_telemetry_hourly` from `silver.fact_sensor_streams`, one row per equipment per hour (`equipment_id`, `date_key`, `hour_of_day`). Load pattern: incremental hourly aggregation and pivot.

* It finds the (equipment, hour) buckets touched by silver rows newer than the watermark (`MAX(gold_load_ts)`), then recomputes those whole buckets from silver and MERGEs them on `telemetry_hour_sk`. This keeps late-arriving readings correct.
* DQ-G05 rejects readings with a NULL timestamp or equipment and asserts `hour_of_day BETWEEN 0 AND 23 AND total_readings_count > 0`.
* Sensor mapping note: the STTM uses literals `'temperature'`, `'vibration'` and `'power'`, but silver currently holds `Temperature_C`, `Spindle_Speed_RPM`, `Hydraulic_Pressure_PSI`, `Vacuum_Pressure_Bar` and `Tonnage_Force`. The pivot therefore matches case-insensitively on the sensor family (`temperature%`, `%vibration%`, `%power%`). Vibration and power columns stay NULL until such sensors appear in silver.

In [0]:
%sql
CREATE TABLE IF NOT EXISTS amperedrive_giga.gold.fact_equipment_telemetry_hourly (
  telemetry_hour_sk    BIGINT         NOT NULL COMMENT 'PK - xxhash64(equipment_id_yyyyMMddHH)',
  date_key             INT            NOT NULL COMMENT 'FK -> dim_date.date_key',
  hour_of_day          INT            NOT NULL COMMENT 'NK - 0..23',
  equipment_sk         BIGINT         NOT NULL COMMENT 'FK -> dim_equipment',
  total_readings_count BIGINT         NOT NULL,
  avg_temperature_c    DECIMAL(38,18),
  max_temperature_c    DECIMAL(38,18),
  avg_vibration_mm_s   DECIMAL(38,18),
  max_vibration_mm_s   DECIMAL(38,18),
  avg_power_kw         DECIMAL(38,18),
  gold_load_ts         TIMESTAMP      NOT NULL,
  CONSTRAINT pk_fact_equipment_telemetry_hourly PRIMARY KEY (telemetry_hour_sk)
)
COMMENT 'Gold hourly equipment telemetry aggregate (pivoted by sensor type) from silver.fact_sensor_streams'

In [0]:
%sql
DECLARE OR REPLACE VARIABLE v_watermark TIMESTAMP;
SET VAR v_watermark = (
  SELECT COALESCE(MAX(gold_load_ts), TIMESTAMP'1900-01-01 00:00:00')
  FROM amperedrive_giga.gold.fact_equipment_telemetry_hourly
);
SELECT v_watermark AS watermark;

In [0]:
%sql
-- DQ-G05: Reject Invalid Timestamp - readings without a timestamp or equipment cannot be bucketed
SELECT COUNT(*) AS dq_g05_rejected_readings
FROM amperedrive_giga.silver.fact_sensor_streams
WHERE updated_at > v_watermark
  AND (timestamp IS NULL OR equipment_id IS NULL)

In [0]:
%sql
CREATE OR REPLACE TEMPORARY VIEW vw_src_fact_equipment_telemetry_hourly AS
WITH valid_readings AS (
  SELECT
    equipment_id,
    timestamp,
    LOWER(TRIM(sensor_type))                 AS sensor_type_norm,
    sensor_value,
    updated_at,
    xxhash64(CONCAT(equipment_id, '_', DATE_FORMAT(timestamp, 'yyyyMMddHH'))) AS telemetry_hour_sk
  FROM amperedrive_giga.silver.fact_sensor_streams
  WHERE timestamp IS NOT NULL
    AND equipment_id IS NOT NULL
),
-- Hour buckets touched by new/changed silver readings since the last gold load
impacted_buckets AS (
  SELECT DISTINCT telemetry_hour_sk
  FROM valid_readings
  WHERE updated_at > v_watermark
)
SELECT
  r.telemetry_hour_sk,
  CAST(DATE_FORMAT(MIN(r.timestamp), 'yyyyMMdd') AS INT)                                AS date_key,
  HOUR(MIN(r.timestamp))                                                                AS hour_of_day,
  xxhash64(CAST(r.equipment_id AS STRING))                                              AS equipment_sk,
  COUNT(1)                                                                              AS total_readings_count,
  CAST(AVG(CASE WHEN r.sensor_type_norm LIKE 'temperature%' THEN r.sensor_value END) AS DECIMAL(38,18)) AS avg_temperature_c,
  CAST(MAX(CASE WHEN r.sensor_type_norm LIKE 'temperature%' THEN r.sensor_value END) AS DECIMAL(38,18)) AS max_temperature_c,
  CAST(AVG(CASE WHEN r.sensor_type_norm LIKE '%vibration%'  THEN r.sensor_value END) AS DECIMAL(38,18)) AS avg_vibration_mm_s,
  CAST(MAX(CASE WHEN r.sensor_type_norm LIKE '%vibration%'  THEN r.sensor_value END) AS DECIMAL(38,18)) AS max_vibration_mm_s,
  CAST(AVG(CASE WHEN r.sensor_type_norm LIKE '%power%'      THEN r.sensor_value END) AS DECIMAL(38,18)) AS avg_power_kw,
  CURRENT_TIMESTAMP()                                                                   AS gold_load_ts
FROM valid_readings r
LEFT SEMI JOIN impacted_buckets b
  ON r.telemetry_hour_sk = b.telemetry_hour_sk
GROUP BY r.telemetry_hour_sk, r.equipment_id, DATE_TRUNC('HOUR', r.timestamp);

SELECT COUNT(*) AS incoming_hour_buckets FROM vw_src_fact_equipment_telemetry_hourly;

In [0]:
%sql
-- DQ-G05: hour_of_day BETWEEN 0 AND 23 AND total_readings_count > 0
SELECT
  COUNT(*) AS invalid_buckets,
  CASE WHEN COUNT(*) > 0
       THEN raise_error(CONCAT('DQ-G05 failed: ', COUNT(*), ' invalid hourly buckets'))
  END AS dq_g05_status
FROM vw_src_fact_equipment_telemetry_hourly
WHERE NOT (hour_of_day BETWEEN 0 AND 23 AND total_readings_count > 0)
   OR hour_of_day IS NULL

In [0]:
%sql
MERGE INTO amperedrive_giga.gold.fact_equipment_telemetry_hourly AS t
USING vw_src_fact_equipment_telemetry_hourly AS s
ON t.telemetry_hour_sk = s.telemetry_hour_sk
WHEN MATCHED AND (
     NOT (t.total_readings_count <=> s.total_readings_count)
  OR NOT (t.avg_temperature_c    <=> s.avg_temperature_c)
  OR NOT (t.max_temperature_c    <=> s.max_temperature_c)
  OR NOT (t.avg_vibration_mm_s   <=> s.avg_vibration_mm_s)
  OR NOT (t.max_vibration_mm_s   <=> s.max_vibration_mm_s)
  OR NOT (t.avg_power_kw         <=> s.avg_power_kw)
) THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *

In [0]:
from delta.tables import DeltaTable

source_table = "amperedrive_giga.silver.fact_sensor_streams"
staged_view = "vw_src_fact_equipment_telemetry_hourly"
target_fqn = "amperedrive_giga.gold.fact_equipment_telemetry_hourly"

rows_extracted = spark.table(source_table).count()  # raw sensor readings in silver source
rows_loaded = spark.table(staged_view).count()      # hourly buckets staged for the MERGE

# MERGE metrics: _sqldf holds the result of the MERGE cell above; fall back to the latest MERGE in Delta history
try:
    m = _sqldf.first().asDict()
    rows_inserted, rows_updated = int(m["num_inserted_rows"]), int(m["num_updated_rows"])
except Exception:
    h = (DeltaTable.forName(spark, target_fqn).history(10)
         .filter("operation = 'MERGE'").orderBy("version", ascending=False).first())
    om = h["operationMetrics"] if h else {}
    rows_inserted = int(om.get("numTargetRowsInserted", 0))
    rows_updated = int(om.get("numTargetRowsUpdated", 0))

result = {
    "rows_extracted": str(rows_extracted),
    "rows_loaded": str(rows_loaded),
    "rows_inserted": str(rows_inserted),
    "rows_updated": str(rows_updated)
}

dbutils.notebook.exit(result)